In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 
mart_schema = get_mart_schema() 

# -------------------------------
# Create mart Table (Environment-aware)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{features_schema}.features_product AS

WITH sales_base AS (
    SELECT
        s.product_id,
        o.order_date,                -- ✅ FIXED
        s.sales AS total_sales,
        s.profit AS total_profit,
        s.discount AS avg_discount
    FROM {catalog}.{gold_schema}.facts_sales s
    JOIN {catalog}.{gold_schema}.facts_orders o
        ON s.order_id = o.order_id
),

-- 2. Sales in last 7 & 30 days
sales_window AS (
    SELECT
        product_id,
        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 7) THEN total_sales ELSE 0 END) AS sales_last_7days,
        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN total_sales ELSE 0 END) AS sales_last_30days
    FROM sales_base
    GROUP BY product_id
),

-- 3. Sales Trend
sales_trend AS (
    SELECT
        product_id,
        CASE WHEN sales_last_30days = 0 THEN 0
             ELSE ROUND((sales_last_7days / 7.0) / (sales_last_30days / 30.0), 2)
        END AS sales_trend
    FROM sales_window
),

-- 4. Discount Sensitivity
discount_sensitivity AS (
    SELECT
        product_id,
        ROUND(AVG(avg_discount) * 100, 2) AS discount_sensitivity
    FROM sales_base
    GROUP BY product_id
),

-- 5. Profit Volatility
profit_volatility AS (
    SELECT
        product_id,
        ROUND(VAR_SAMP(total_profit), 2) AS profit_volatility
    FROM sales_base
    GROUP BY product_id
),

-- 6. Seasonality Index
seasonality AS (
    SELECT
        product_id,
        ROUND(AVG(monthly_sales) / NULLIF(AVG(overall_avg_sales), 0), 2) AS seasonality_index
    FROM (
        SELECT
            product_id,
            MONTH(order_date) AS month,
            AVG(total_sales) AS monthly_sales,
            AVG(AVG(total_sales)) OVER (PARTITION BY product_id) AS overall_avg_sales
        FROM sales_base
        GROUP BY product_id, MONTH(order_date)
    ) t
    GROUP BY product_id
)

SELECT
    s.product_id,
    CURRENT_DATE() AS feature_date,
    s.sales_last_7days,
    s.sales_last_30days,
    st.sales_trend,
    ds.discount_sensitivity,
    pv.profit_volatility,
    se.seasonality_index
FROM sales_window s
JOIN sales_trend st USING (product_id)
JOIN discount_sensitivity ds USING (product_id)
JOIN profit_volatility pv USING (product_id)
JOIN seasonality se USING (product_id);
""")
